# BODIK と国土地理院（GSI）の指定緊急避難場所データの比較実験

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YanTKYS/sheltermatch/blob/main/experiments/gsi_shelter_compare/gsi_shelter_compare.ipynb)

`sheltermatch` が現在使っている **BODIK の指定緊急避難場所データ（設定ファイルで指定した自治体。糸満市が既定）** と、国土地理院が地理院タイルとして公開している
**指定緊急避難場所データ（`skhb01`〜`skhb08`）** を比べ、件数・施設・名称・住所・座標にどの程度差があるかを確認する実験です。
**上のセルから順に実行**すれば、取得 → 重複整理 → 比較 → 集計表示 → CSV確認・ダウンロードまで完結します。

* **GSI を正解データとして BODIK を評価するものではありません。** 2つの公開データの差異を把握するための比較です。
  どちらが正しい・どちらへ切り替えるべき、という判断はこのNotebookでは行いません（表示するのは事実の整理だけです）。
* 対象の自治体（自治体名・BODIK の resource_id）と比較の設定は、設定ファイル `configs/<CONFIG_NAME>.json`
  （糸満市は `itoman-city`）から読み込みます。通常は Notebook を編集する必要はありません。
* 本番 `sheltermatch` のデータソースを変更するものではありません（本番の `sheltermatch.ipynb`・`src/` とは独立しています）。
* 使うのは**公開データだけ**（BODIK Data API・地理院タイル）です。**要支援者データ等の個人情報は使いません。**
* 国土地理院の公式ページによると、GSI のデータは市町村が登録した情報で、最新でない場合や未掲載の場合があり、
  最新かつ詳細な状況は当該市町村への確認が必要です。

比較の考え方（正規化・マッチングのルール）や公式資料のリンクは、同じフォルダの
[README](https://github.com/YanTKYS/sheltermatch/blob/main/experiments/gsi_shelter_compare/README.md) を参照してください。

In [ ]:
# ===== 利用者設定 =====
# 通常は、このセルを変更する必要はありません（糸満市の通常利用では、そのまま上から実行してください）。

# 自治体ごとの設定は、設定ファイル configs/<CONFIG_NAME>.json にまとめています（糸満市は "itoman-city"）。
# 別の自治体で使うときだけ、configs/ にあるその自治体の設定名へ変更してください。
CONFIG_NAME = "itoman-city"

# 以下は、このNotebookの既定値です。設定ファイルに同じ項目があるときは、設定ファイルの値が使われます
# （実行時に、最終的な値と設定元を「設定ファイル読込」セルで表示します）。設定ファイルに項目が無いときだけ、
# ここの値を使います。

# BODIK の各施設の座標を含む z=10 タイルの「周囲に追加で取得するタイル数」。
# 1 なら周囲1枚ぶん（3×3）を取得し、BODIK に無い近隣の GSI 施設も取り逃しにくくなります。0 なら座標を含むタイルだけ。
TILE_RING = 1

# 名称・住所が一致する組のうち、座標差がこの距離（m）以内のものだけを exact_match（同一施設）とします。
# これを超える組は review_needed（人が確認する）になります。正誤の判定基準ではなく、目視確認の目安です。
EXACT_MAX_DISTANCE_M = 30

# 名称が異なっていても、BODIK と GSI の施設がこの距離（m）以内にあれば review_needed として挙げます。
NEAR_DISTANCE_M = 100

# 結果CSV（gsi_bodik_comparison.csv / gsi_unique_shelters.csv）を保存するフォルダ。
OUTPUT_DIR = "output"

print("利用者設定（Notebookの既定値）を読み込みました。")
print(f"  CONFIG_NAME = '{CONFIG_NAME}'")
print(f"  OUTPUT_DIR  = {OUTPUT_DIR!r}")
print("  比較の設定（TILE_RING 等）の最終的な値は、「設定ファイル読込」セルで表示します。")

In [ ]:
# ===== 実行環境準備 =====
# 必要なライブラリを導入します（Google Colab ではほとんど導入済みです）。この実験で使うのは次の3つだけです。
%pip install -q requests geopy pandas

import importlib.util
import sys
from pathlib import Path

import pandas as pd
import requests
from IPython.display import display

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"実行環境: {'Google Colab' if IN_COLAB else 'Google Colab 以外（ローカル等）'}")

In [ ]:
# ===== 比較ロジックの読み込み =====
# 比較のロジック（正規化・GSI内の重複整理・突き合わせ・集計）は compare_logic.py に、BODIK・GSI の取得とCSV出力の
# 関数は compare.py に、設定ファイルの読込・検証は src/config/municipality_config.py にあり、このNotebookは
# それらを読み込んで使います（ロジックをNotebookへコピーして二重管理しません）。
# このリポジトリを開いている場合（現在のフォルダかその上位に src/ と experiments/ がある場合）はそれを使い、
# 無ければ GitHub から取得します。取得先の版は、次の COMPARE_LOGIC_REF の1か所だけで決まります
# （通常は main のままで構いません）。
COMPARE_LOGIC_REF = "main"

RAW_REPO_URL = f"https://raw.githubusercontent.com/YanTKYS/sheltermatch/{COMPARE_LOGIC_REF}"
EXPERIMENT_DIR_PATH = "experiments/gsi_shelter_compare"
MODULE_FILES = (
    f"{EXPERIMENT_DIR_PATH}/compare_logic.py",
    f"{EXPERIMENT_DIR_PATH}/compare.py",
    "src/config/municipality_config.py",
)


def find_local_repo_root():
    for candidate in (Path.cwd(), *list(Path.cwd().parents)[:3]):
        if all((candidate / name).is_file() for name in MODULE_FILES):
            return candidate
    return None


repo_root = find_local_repo_root()
if repo_root is None:
    repo_root = Path("gsi_shelter_compare_modules")
    for name in MODULE_FILES:
        response = requests.get(f"{RAW_REPO_URL}/{name}", timeout=30)
        response.raise_for_status()
        (repo_root / name).parent.mkdir(parents=True, exist_ok=True)
        (repo_root / name).write_bytes(response.content)
    logic_source = f"{COMPARE_LOGIC_REF}（GitHubから取得）"
else:
    logic_source = f"ローカルの compare_logic.py（{repo_root.resolve() / EXPERIMENT_DIR_PATH}）"

# セルを再実行したときや、別の場所のモジュールを読み込み済みのときに、古いものが残らないようにする
for name in ("compare", "compare_logic", "municipality_config"):
    sys.modules.pop(name, None)
sys.path.insert(0, str((repo_root / EXPERIMENT_DIR_PATH).resolve()))
import compare_logic as cl
import compare

config_module_path = repo_root / "src" / "config" / "municipality_config.py"
config_spec = importlib.util.spec_from_file_location("municipality_config", config_module_path)
municipality_config = importlib.util.module_from_spec(config_spec)
config_spec.loader.exec_module(municipality_config)

print(f"比較ロジック: {logic_source}")
print("比較ロジックを読み込みました。")

In [ ]:
# ===== 設定ファイル読込 =====
# 自治体ごとの設定ファイル configs/<CONFIG_NAME>.json を読み込み、比較の設定へ反映します。
# - 設定ファイルに値がある項目は、設定ファイルの値を使います（上の「利用者設定」セルの値より優先）。
# - 設定ファイルに該当する項目が無いときだけ、「利用者設定」セルのNotebook既定値を使います。
# - 設定ファイルそのものを取得・解釈できないとき（見つからない・通信できない・JSONの構文が不正・
#   必須項目が無い・型や値が不正）は、Notebookの既定値で続行せず、原因を表示して止まります。
# 同じ名前の設定ファイルが、このリポジトリの configs/ にあればそれを使い、無ければ、比較ロジックと同じ版
# （COMPARE_LOGIC_REF）のGitHubから取得します。

config, config_origin = municipality_config.load_config(CONFIG_NAME, RAW_REPO_URL)

TILE_RING, tile_ring_source = municipality_config.resolve_setting(
    config, "gsi_shelter_compare", "tile_ring", TILE_RING)
EXACT_MAX_DISTANCE_M, exact_max_distance_source = municipality_config.resolve_setting(
    config, "gsi_shelter_compare", "exact_max_distance_m", EXACT_MAX_DISTANCE_M)
NEAR_DISTANCE_M, near_distance_source = municipality_config.resolve_setting(
    config, "gsi_shelter_compare", "near_distance_m", NEAR_DISTANCE_M)

# 設定ファイルの値は検証済み。Notebookの既定値を書き換えた場合に備えて、最終的な値も確認する。
if isinstance(TILE_RING, bool) or not isinstance(TILE_RING, int) or TILE_RING < 0:
    raise ValueError(f"TILE_RING は0以上の整数を指定してください。現在の値: {TILE_RING!r}")
for name, value in (("EXACT_MAX_DISTANCE_M", EXACT_MAX_DISTANCE_M), ("NEAR_DISTANCE_M", NEAR_DISTANCE_M)):
    if isinstance(value, bool) or not isinstance(value, (int, float)) or value <= 0:
        raise ValueError(f"{name} は0より大きい数値を指定してください。現在の値: {value!r}")

# 自治体固有の値（設定ファイルの必須項目）。以降のセルで使います。
MUNICIPALITY_NAME = config["municipality"]["name"]
BODIK_RESOURCE_ID = config["bodik"]["resource_id"]

print("\n".join(municipality_config.describe_municipality(config, CONFIG_NAME, config_origin)))
print()
print("比較設定:")
print("\n".join(municipality_config.describe_settings([
    ("TILE_RING", TILE_RING, tile_ring_source),
    ("EXACT_MAX_DISTANCE_M", EXACT_MAX_DISTANCE_M, exact_max_distance_source),
    ("NEAR_DISTANCE_M", NEAR_DISTANCE_M, near_distance_source),
])))

In [ ]:
# ===== BODIK取得 =====
# sheltermatch.ipynb（避難所取得セル）と同じ BODIK Data API から、設定ファイルの bodik.resource_id の
# データセット（指定緊急避難場所）を全件取得します。比較に使うのは 名称・住所・緯度・経度 です。
session = requests.Session()
session.headers["User-Agent"] = compare.USER_AGENT

print(f"BODIK Data API から取得します（{MUNICIPALITY_NAME} / resource_id: {BODIK_RESOURCE_ID}）…")
bodik = compare.bodik_facilities(compare.fetch_bodik_records(session, BODIK_RESOURCE_ID))
bodik_with_coords = [b for b in bodik if b.latitude is not None]
if not bodik_with_coords:
    raise RuntimeError("座標のあるBODIK施設が0件のため、GSIの取得範囲を決められません。")

print(f"BODIK施設数: {len(bodik)}件")
print(f"座標あり: {len(bodik_with_coords)}件")
print(f"座標なし: {len(bodik) - len(bodik_with_coords)}件")

bodik_df = pd.DataFrame({
    "name": [b.original_name for b in bodik],
    "address": [b.original_address for b in bodik],
    "latitude": [b.latitude for b in bodik],
    "longitude": [b.longitude for b in bodik],
})
display(bodik_df.head(10))

In [ ]:
# ===== GSI取得タイルの確認 =====
# 日本全国のタイルは取得しません。BODIK の各施設の座標を含む z=10 タイルと、その周囲 TILE_RING 枚ぶんの隣接タイルだけを
# 重複なしで取得します。実際に取得する前に、タイル数とリクエスト数を確認できます。
tiles = cl.tiles_for_points([(b.latitude, b.longitude) for b in bodik_with_coords], compare.GSI_ZOOM, TILE_RING)
layers = list(cl.GSI_LAYERS)

display(pd.DataFrame({
    "項目": ["ズームレベル", "取得タイル数", "対象レイヤー数", "想定リクエスト数（タイル数×レイヤー数）"],
    "値": [f"z={compare.GSI_ZOOM}", len(tiles), len(layers), len(tiles) * len(layers)],
}))
display(pd.DataFrame({"レイヤー": layers, "災害種別": [cl.GSI_LAYERS[layer] for layer in layers]}))
print("※ データの無いタイルは 404 が返りますが、取得失敗ではなく「データなし」として扱います。")

In [ ]:
# ===== GSI取得 =====
# レイヤーごとに、上で確認したタイルを取得します（1リクエストごとに0.2秒あけます。通常は1〜2分ほどです）。
print("GSIデータを取得しています...")
fetched = []
tile_layer_hits = 0
for layer in layers:
    layer_fetched, hits = compare.fetch_gsi_features(session, tiles, layers=(layer,))
    fetched.extend(layer_fetched)
    tile_layer_hits += hits
    print(f"{layer} 完了（{len(layer_fetched)}件）")

gsi_features, skipped_features = [], 0
for layer, feature in fetched:
    facility = cl.gsi_facility_from_feature(layer, feature)
    if facility is None:
        skipped_features += 1
    else:
        gsi_features.append(facility)

data_tiles = {cl.tile_xy(f.latitude, f.longitude, compare.GSI_ZOOM) for f in gsi_features if f.latitude is not None}
print()
print(f"取得Feature数（災害種別レイヤー重複込み・住所での絞り込み前）: {len(fetched)}件")
print(f"データの存在したタイル数: {len(data_tiles)}枚（レイヤーごとに数えた延べ {tile_layer_hits}枚）")
if skipped_features:
    print(f"Point以外のため読み飛ばしたFeature: {skipped_features}件")

In [ ]:
# ===== GSI内の重複整理・対象範囲の確認 =====
# GSI は災害種別ごとにレイヤーが分かれているため、同じ施設が複数レイヤーに出てきます。名称・住所（正規化後）が一致し、
# 座標差が5m以内のものだけを1施設へ統合します（元のレイヤーは gsi_layers に残ります）。
# そのあと、住所などから比較の対象にする施設を scope で振り分けます（住所に自治体名が無いだけでは除外しません）。
unique_gsi, split_count = cl.merge_gsi_layers(gsi_features)
cl.assign_gsi_scope(unique_gsi, bodik, MUNICIPALITY_NAME, near_m=NEAR_DISTANCE_M)
compared_gsi = [g for g in unique_gsi if g.scope in cl.COMPARED_SCOPES]

display(pd.DataFrame({
    "項目": ["GSI Feature数（レイヤー重複込み）", "GSIユニーク施設数（重複整理後・絞り込み前）", "比較対象施設数"],
    "件数": [len(gsi_features), len(unique_gsi), len(compared_gsi)],
}))
if split_count:
    print(f"名称・住所が同じでも座標が離れていたため統合しなかった施設: {split_count}件")

SCOPE_LABELS = {
    "in_city": "住所に自治体名（設定ファイルの municipality.name）を含む（比較対象）",
    "address_missing_near": "住所が空欄で、BODIKの座標範囲の近く（比較対象。住所は比較できない）",
    "other_address_near_bodik": "住所に自治体名は無いが、BODIKと同名または近接（比較対象。住所が異なるので review_needed）",
    "excluded_address_missing_far": "住所が空欄で、範囲の外（比較対象外）",
    "excluded_other_address": "上のいずれでもない（他市町村の施設など。比較対象外）",
}
scope_counts = pd.Series([g.scope for g in unique_gsi], dtype=object).value_counts()
display(pd.DataFrame({
    "scope": list(SCOPE_LABELS),
    "件数": [int(scope_counts.get(scope, 0)) for scope in SCOPE_LABELS],
    "説明": list(SCOPE_LABELS.values()),
}))


def gsi_dataframe(facilities):
    return pd.DataFrame({
        "scope": [g.scope for g in facilities],
        "gsi_name": [g.original_name for g in facilities],
        "gsi_address": [g.original_address for g in facilities],
        "gsi_latitude": [g.latitude for g in facilities],
        "gsi_longitude": [g.longitude for g in facilities],
        "gsi_layers": [";".join(g.gsi_layers) for g in facilities],
    })


# 比較対象外になった施設の例（他市町村の施設などが並ぶのが通常です。対象の自治体の施設が混ざっていないか確認できます）
excluded_gsi = [g for g in unique_gsi if g.scope not in cl.COMPARED_SCOPES]
print(f"\n比較対象外になった施設（先頭10件 / 全{len(excluded_gsi)}件）")
display(gsi_dataframe(excluded_gsi).head(10))

In [ ]:
# ===== BODIK と GSI の比較 =====
# exact_match（名称・住所が一致し座標差も小さい）だけを同一施設と断定し、曖昧なものは review_needed にします。
# 比較の詳しいルールは README を参照してください。
rows = cl.compare_facilities(bodik, compared_gsi, EXACT_MAX_DISTANCE_M, NEAR_DISTANCE_M)
summary = cl.summarize(rows)

STATUSES = ["exact_match", "review_needed", "bodik_only", "gsi_only"]


def rows_dataframe(rows):
    def attr(obj, name):
        return getattr(obj, name) if obj is not None else None
    return pd.DataFrame({
        "status": [r.status for r in rows],
        "reason": [r.reason for r in rows],
        "bodik_name": [attr(r.bodik, "original_name") for r in rows],
        "bodik_address": [attr(r.bodik, "original_address") for r in rows],
        "bodik_latitude": [attr(r.bodik, "latitude") for r in rows],
        "bodik_longitude": [attr(r.bodik, "longitude") for r in rows],
        "gsi_name": [attr(r.gsi, "original_name") for r in rows],
        "gsi_address": [attr(r.gsi, "original_address") for r in rows],
        "gsi_latitude": [attr(r.gsi, "latitude") for r in rows],
        "gsi_longitude": [attr(r.gsi, "longitude") for r in rows],
        "distance_m": [None if r.distance_m is None else round(r.distance_m, 1) for r in rows],
        "gsi_layers": [";".join(r.gsi.gsi_layers) if r.gsi is not None else "" for r in rows],
        "notes": [" | ".join(r.notes) for r in rows],
    })


def show_all(df, columns):
    """件数が多くても省略せず、全行・全文字を表示する。"""
    if df.empty:
        print("該当なし")
        return
    with pd.option_context("display.max_rows", None, "display.max_colwidth", None, "display.width", 250):
        display(df[columns].reset_index(drop=True))


result_df = rows_dataframe(rows)
status_counts = result_df["status"].value_counts()
display(pd.DataFrame({"status": STATUSES, "件数": [int(status_counts.get(s, 0)) for s in STATUSES]}))
print(f"BODIK {len(bodik)}件 / GSI比較対象 {len(compared_gsi)}件")
print("※ review_needed の行には、同じ施設が他の行にも現れる「1対多」の行を含みます（notes に記載）。")

COMPARE_COLUMNS = ["bodik_name", "bodik_address", "gsi_name", "gsi_address", "distance_m", "gsi_layers", "notes"]

print("\n【exact_match】（先頭10件）")
exact_df = result_df[result_df["status"] == "exact_match"]
show_all(exact_df.head(10), COMPARE_COLUMNS)

In [ ]:
# ===== review_needed（人が確認したい組）=====
# 今回の比較で最も確認したい部分です。全件を表示します。
review_df = result_df[result_df["status"] == "review_needed"].sort_values(["reason", "bodik_name"])

print("【理由別の件数】")
review_reason_counts = review_df["reason"].value_counts().rename_axis("reason").reset_index(name="件数")
show_all(review_reason_counts, ["reason", "件数"])

print(f"\n【review_needed 全{len(review_df)}件】")
show_all(review_df, ["reason"] + COMPARE_COLUMNS)

In [ ]:
# ===== review_needed を理由別に絞り込む =====
# 上の「理由別の件数」の reason を1つ選んで、その組だけを表示します。
# 既定では件数の最も多い理由を表示します。ほかの理由を見るときは、下の1行目を書き換えてください。
# 例: reason_to_check = "名称は一致するが住所が異なる"
reason_to_check = review_reason_counts["reason"].iloc[0] if len(review_reason_counts) else ""

print(f"reason: {reason_to_check or '（review_needed なし）'}")
show_all(review_df[review_df["reason"] == reason_to_check], COMPARE_COLUMNS)

In [ ]:
# ===== bodik_only（BODIKにあり、GSI側に対応施設が見つからないもの）=====
# notes の「参考」は最寄りのGSI施設です（同一施設という意味ではありません）。
bodik_only_df = result_df[result_df["status"] == "bodik_only"]
print(f"bodik_only: {len(bodik_only_df)}件")
show_all(bodik_only_df, ["bodik_name", "bodik_address", "bodik_latitude", "bodik_longitude", "notes"])

In [ ]:
# ===== gsi_only（GSIにあり、BODIK側に対応施設が見つからないもの）=====
# BODIKとGSIの施設数の差を調べるための一覧です。全件を表示します（件数は公開データの更新で変わることがあります）。
gsi_only_df = result_df[result_df["status"] == "gsi_only"].sort_values("gsi_name")
print(f"gsi_only: {len(gsi_only_df)}件")
show_all(gsi_only_df, ["gsi_name", "gsi_address", "gsi_latitude", "gsi_longitude", "gsi_layers", "notes"])

In [ ]:
# ===== 座標差・名称と住所の表記差 =====
# 座標差は、BODIK の座標と GSI の座標の測地線距離（geopy.distance.geodesic）です。閾値は目視確認の目安で、正誤の判定ではありません。
def distance_table(named_stats):
    records = []
    for label, stats in named_stats:
        if stats is None:
            records.append({"対象": label, "件数": 0})
            continue
        record = {"対象": label, "件数": stats["count"], "最大(m)": round(stats["max"], 1),
                  "中央値(m)": round(stats["median"], 1), "平均(m)": round(stats["mean"], 1)}
        record.update({f"{t}m超": n for t, n in stats["over"].items()})
        records.append(record)
    return pd.DataFrame(records)


print("【座標差】")
display(distance_table([
    ("exact_match", summary["exact_distance"]),
    ("exact_match + review_needed", summary["paired_distance"]),
]))

print("\n【名称・住所の表記差】（対応付けできた組。元の文字列を比べた結果）")
display(pd.DataFrame([
    {"対象": label,
     "元の表記が同一": summary[key]["identical"],
     "正規化後にだけ一致": summary[key]["identical_after_normalization_only"],
     "異なる": summary[key]["different"]}
    for label, key in (("exact_match の名称", "exact_name"), ("exact_match の住所", "exact_address"),
                       ("exact_match + review_needed の名称", "paired_name"),
                       ("exact_match + review_needed の住所", "paired_address"))
]))
print("※ 正規化は NFKC・ハイフン類の統一・空白の整理だけです。語句の削除や意味的な変換はしていません。")

In [ ]:
# ===== CSV出力 =====
# 比較結果（gsi_bodik_comparison.csv）と、レイヤー間の重複を整理した GSI の施設一覧（gsi_unique_shelters.csv）を保存します。
# 比較対象外の施設も、gsi_unique_shelters.csv の scope 列で確認できます。
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
comparison_csv = output_dir / "gsi_bodik_comparison.csv"
unique_csv = output_dir / "gsi_unique_shelters.csv"

compare.write_comparison_csv(comparison_csv, rows)
compare.write_gsi_unique_csv(unique_csv, unique_gsi)

print("CSVを作成しました。")
print(f"  {comparison_csv}（{len(rows)}行）")
print(f"  {unique_csv}（{len(unique_gsi)}行）")
print("※ 取得した公開データから作った結果です。リポジトリへコミットしないでください。")

In [ ]:
# ===== CSVのダウンロード =====
# Google Colab ではブラウザへ2ファイルをダウンロードします。Colab 以外では、保存済みのファイルを確認するだけです。
if IN_COLAB:
    files.download(str(comparison_csv))
    files.download(str(unique_csv))
else:
    print("Google Colab ではないため、ダウンロードは行いません。次のファイルに保存済みです。")
    print(f"  {comparison_csv.resolve()}")
    print(f"  {unique_csv.resolve()}")